<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/atividade2_biblioteca_universitaria_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 2 – Biblioteca Universitária
Este notebook implementa o banco completo no MariaDB, incluindo livros, autores, editoras, categorias, usuários, empréstimos, devoluções, multas e pagamentos parcelados.

In [1]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y mariadb-server mariadb-client >/dev/null
mkdir -p /run/mysqld
chown mysql:mysql /run/mysqld
service mariadb start >/dev/null 2>&1 || true

for i in {1..20}; do
  if mariadb-admin -u root ping --silent; then
    mariadb --version
    echo 'MariaDB iniciado com sucesso.'
    exit 0
  fi
  sleep 1
done

echo 'Não foi possível iniciar o MariaDB.'
exit 1


mysqld is alive
mariadb  Ver 15.1 Distrib 10.6.23-MariaDB, for debian-linux-gnu (x86_64) using  EditLine wrapper
MariaDB iniciado com sucesso.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
from pathlib import Path

sql = "-- ============================================================\n-- ATIVIDADE 2 – SISTEMA DE BIBLIOTECA UNIVERSITÁRIA\n-- IMPLEMENTAÇÃO COMPLETA EM MYSQL/MARIADB\n-- ============================================================\n\nDROP DATABASE IF EXISTS biblioteca_universitaria;\n\nCREATE DATABASE biblioteca_universitaria\n    CHARACTER SET utf8mb4\n    COLLATE utf8mb4_unicode_ci;\n\nUSE biblioteca_universitaria;\n\n-- ============================================================\n-- 1. TABELAS\n-- ============================================================\n\nCREATE TABLE editora (\n    id_editora INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n    cidade VARCHAR(100),\n    telefone VARCHAR(20),\n\n    CONSTRAINT pk_editora\n        PRIMARY KEY (id_editora),\n\n    CONSTRAINT uq_editora_nome\n        UNIQUE (nome)\n) ENGINE = InnoDB;\n\nCREATE TABLE categoria (\n    id_categoria INT AUTO_INCREMENT,\n    nome VARCHAR(100) NOT NULL,\n    descricao VARCHAR(255),\n\n    CONSTRAINT pk_categoria\n        PRIMARY KEY (id_categoria),\n\n    CONSTRAINT uq_categoria_nome\n        UNIQUE (nome)\n) ENGINE = InnoDB;\n\nCREATE TABLE autor (\n    id_autor INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n    nacionalidade VARCHAR(80),\n    data_nascimento DATE,\n\n    CONSTRAINT pk_autor\n        PRIMARY KEY (id_autor)\n) ENGINE = InnoDB;\n\nCREATE TABLE livro (\n    id_livro INT AUTO_INCREMENT,\n    titulo VARCHAR(200) NOT NULL,\n    isbn VARCHAR(20) NOT NULL,\n    ano_publicacao YEAR NOT NULL,\n    quantidade_total INT NOT NULL,\n    quantidade_disponivel INT NOT NULL,\n    id_editora INT NOT NULL,\n    id_categoria INT NOT NULL,\n\n    CONSTRAINT pk_livro\n        PRIMARY KEY (id_livro),\n\n    CONSTRAINT uq_livro_isbn\n        UNIQUE (isbn),\n\n    CONSTRAINT chk_livro_quantidade_total\n        CHECK (quantidade_total >= 0),\n\n    CONSTRAINT chk_livro_quantidade_disponivel\n        CHECK (\n            quantidade_disponivel >= 0\n            AND quantidade_disponivel <= quantidade_total\n        ),\n\n    CONSTRAINT fk_livro_editora\n        FOREIGN KEY (id_editora)\n        REFERENCES editora(id_editora)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT,\n\n    CONSTRAINT fk_livro_categoria\n        FOREIGN KEY (id_categoria)\n        REFERENCES categoria(id_categoria)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT\n) ENGINE = InnoDB;\n\nCREATE TABLE livro_autor (\n    id_livro INT NOT NULL,\n    id_autor INT NOT NULL,\n\n    CONSTRAINT pk_livro_autor\n        PRIMARY KEY (\n            id_livro,\n            id_autor\n        ),\n\n    CONSTRAINT fk_livro_autor_livro\n        FOREIGN KEY (id_livro)\n        REFERENCES livro(id_livro)\n        ON UPDATE CASCADE\n        ON DELETE CASCADE,\n\n    CONSTRAINT fk_livro_autor_autor\n        FOREIGN KEY (id_autor)\n        REFERENCES autor(id_autor)\n        ON UPDATE CASCADE\n        ON DELETE CASCADE\n) ENGINE = InnoDB;\n\nCREATE TABLE usuario (\n    id_usuario INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n    email VARCHAR(150) NOT NULL,\n    matricula VARCHAR(30) NOT NULL,\n    tipo_usuario VARCHAR(20) NOT NULL,\n    telefone VARCHAR(20),\n    situacao VARCHAR(20) NOT NULL DEFAULT 'Ativo',\n\n    CONSTRAINT pk_usuario\n        PRIMARY KEY (id_usuario),\n\n    CONSTRAINT uq_usuario_email\n        UNIQUE (email),\n\n    CONSTRAINT uq_usuario_matricula\n        UNIQUE (matricula),\n\n    CONSTRAINT chk_usuario_nome\n        CHECK (CHAR_LENGTH(TRIM(nome)) > 0),\n\n    CONSTRAINT chk_usuario_tipo\n        CHECK (\n            tipo_usuario IN (\n                'Aluno',\n                'Professor'\n            )\n        ),\n\n    CONSTRAINT chk_usuario_situacao\n        CHECK (\n            situacao IN (\n                'Ativo',\n                'Bloqueado',\n                'Inativo'\n            )\n        )\n) ENGINE = InnoDB;\n\nCREATE TABLE emprestimo (\n    id_emprestimo INT AUTO_INCREMENT,\n    id_usuario INT NOT NULL,\n    id_livro INT NOT NULL,\n    data_emprestimo DATE NOT NULL,\n    data_prevista_devolucao DATE NOT NULL,\n    status VARCHAR(20) NOT NULL DEFAULT 'Em andamento',\n\n    CONSTRAINT pk_emprestimo\n        PRIMARY KEY (id_emprestimo),\n\n    CONSTRAINT chk_emprestimo_datas\n        CHECK (\n            data_prevista_devolucao >= data_emprestimo\n        ),\n\n    CONSTRAINT chk_emprestimo_status\n        CHECK (\n            status IN (\n                'Em andamento',\n                'Devolvido',\n                'Atrasado'\n            )\n        ),\n\n    CONSTRAINT fk_emprestimo_usuario\n        FOREIGN KEY (id_usuario)\n        REFERENCES usuario(id_usuario)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT,\n\n    CONSTRAINT fk_emprestimo_livro\n        FOREIGN KEY (id_livro)\n        REFERENCES livro(id_livro)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT\n) ENGINE = InnoDB;\n\nCREATE TABLE devolucao (\n    id_devolucao INT AUTO_INCREMENT,\n    id_emprestimo INT NOT NULL,\n    data_devolucao DATE NOT NULL,\n    observacao VARCHAR(255),\n\n    CONSTRAINT pk_devolucao\n        PRIMARY KEY (id_devolucao),\n\n    CONSTRAINT uq_devolucao_emprestimo\n        UNIQUE (id_emprestimo),\n\n    CONSTRAINT fk_devolucao_emprestimo\n        FOREIGN KEY (id_emprestimo)\n        REFERENCES emprestimo(id_emprestimo)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT\n) ENGINE = InnoDB;\n\nCREATE TABLE multa (\n    id_multa INT AUTO_INCREMENT,\n    id_emprestimo INT NOT NULL,\n    valor_total DECIMAL(10,2) NOT NULL,\n    valor_pago DECIMAL(10,2) NOT NULL DEFAULT 0.00,\n    data_geracao DATE NOT NULL,\n    status VARCHAR(30) NOT NULL DEFAULT 'Pendente',\n\n    CONSTRAINT pk_multa\n        PRIMARY KEY (id_multa),\n\n    CONSTRAINT uq_multa_emprestimo\n        UNIQUE (id_emprestimo),\n\n    CONSTRAINT chk_multa_valor_total\n        CHECK (valor_total > 0),\n\n    CONSTRAINT chk_multa_valor_pago\n        CHECK (\n            valor_pago >= 0\n            AND valor_pago <= valor_total\n        ),\n\n    CONSTRAINT chk_multa_status\n        CHECK (\n            status IN (\n                'Pendente',\n                'Parcialmente paga',\n                'Paga',\n                'Cancelada'\n            )\n        ),\n\n    CONSTRAINT fk_multa_emprestimo\n        FOREIGN KEY (id_emprestimo)\n        REFERENCES emprestimo(id_emprestimo)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT\n) ENGINE = InnoDB;\n\nCREATE TABLE pagamento_multa (\n    id_pagamento INT AUTO_INCREMENT,\n    id_multa INT NOT NULL,\n    data_pagamento DATE NOT NULL,\n    valor_pago DECIMAL(10,2) NOT NULL,\n    forma_pagamento VARCHAR(30) NOT NULL,\n\n    CONSTRAINT pk_pagamento_multa\n        PRIMARY KEY (id_pagamento),\n\n    CONSTRAINT chk_pagamento_valor\n        CHECK (valor_pago > 0),\n\n    CONSTRAINT chk_pagamento_forma\n        CHECK (\n            forma_pagamento IN (\n                'Dinheiro',\n                'PIX',\n                'Cartão',\n                'Transferência'\n            )\n        ),\n\n    CONSTRAINT fk_pagamento_multa\n        FOREIGN KEY (id_multa)\n        REFERENCES multa(id_multa)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT\n) ENGINE = InnoDB;\n\n-- ============================================================\n-- 2. TRIGGERS\n-- ============================================================\n\nDROP TRIGGER IF EXISTS trg_validar_emprestimo;\nDROP TRIGGER IF EXISTS trg_baixar_estoque_emprestimo;\nDROP TRIGGER IF EXISTS trg_validar_devolucao;\nDROP TRIGGER IF EXISTS trg_processar_devolucao;\nDROP TRIGGER IF EXISTS trg_validar_pagamento_multa;\nDROP TRIGGER IF EXISTS trg_atualizar_multa_pagamento;\n\nDELIMITER $$\n\n-- Valida usuário, livro, disponibilidade e datas\nCREATE TRIGGER trg_validar_emprestimo\nBEFORE INSERT ON emprestimo\nFOR EACH ROW\nBEGIN\n    DECLARE v_usuario_existe INT DEFAULT 0;\n    DECLARE v_situacao_usuario VARCHAR(20);\n    DECLARE v_livro_existe INT DEFAULT 0;\n    DECLARE v_quantidade_disponivel INT DEFAULT 0;\n\n    SELECT\n        COUNT(*),\n        MAX(situacao)\n    INTO\n        v_usuario_existe,\n        v_situacao_usuario\n    FROM usuario\n    WHERE id_usuario = NEW.id_usuario;\n\n    IF v_usuario_existe = 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Usuário não encontrado.';\n    END IF;\n\n    IF v_situacao_usuario <> 'Ativo' THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Somente usuários ativos podem realizar empréstimos.';\n    END IF;\n\n    SELECT\n        COUNT(*),\n        COALESCE(MAX(quantidade_disponivel), 0)\n    INTO\n        v_livro_existe,\n        v_quantidade_disponivel\n    FROM livro\n    WHERE id_livro = NEW.id_livro;\n\n    IF v_livro_existe = 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Livro não encontrado.';\n    END IF;\n\n    IF v_quantidade_disponivel <= 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Não há exemplar disponível para empréstimo.';\n    END IF;\n\n    IF NEW.data_prevista_devolucao < NEW.data_emprestimo THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'A data prevista deve ser igual ou posterior à data do empréstimo.';\n    END IF;\nEND$$\n\n-- Reduz a quantidade disponível após o empréstimo\nCREATE TRIGGER trg_baixar_estoque_emprestimo\nAFTER INSERT ON emprestimo\nFOR EACH ROW\nBEGIN\n    UPDATE livro\n    SET quantidade_disponivel =\n        quantidade_disponivel - 1\n    WHERE id_livro = NEW.id_livro;\nEND$$\n\n-- Valida a devolução\nCREATE TRIGGER trg_validar_devolucao\nBEFORE INSERT ON devolucao\nFOR EACH ROW\nBEGIN\n    DECLARE v_emprestimo_existe INT DEFAULT 0;\n    DECLARE v_data_emprestimo DATE;\n    DECLARE v_status VARCHAR(20);\n\n    SELECT\n        COUNT(*),\n        MAX(data_emprestimo),\n        MAX(status)\n    INTO\n        v_emprestimo_existe,\n        v_data_emprestimo,\n        v_status\n    FROM emprestimo\n    WHERE id_emprestimo = NEW.id_emprestimo;\n\n    IF v_emprestimo_existe = 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Empréstimo não encontrado.';\n    END IF;\n\n    IF v_status = 'Devolvido' THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Este empréstimo já foi devolvido.';\n    END IF;\n\n    IF NEW.data_devolucao < v_data_emprestimo THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'A devolução não pode ocorrer antes do empréstimo.';\n    END IF;\nEND$$\n\n-- Atualiza livro e empréstimo e gera multa se houver atraso\nCREATE TRIGGER trg_processar_devolucao\nAFTER INSERT ON devolucao\nFOR EACH ROW\nBEGIN\n    DECLARE v_id_livro INT;\n    DECLARE v_data_prevista DATE;\n    DECLARE v_dias_atraso INT DEFAULT 0;\n    DECLARE v_valor_diario DECIMAL(10,2) DEFAULT 2.00;\n\n    SELECT\n        id_livro,\n        data_prevista_devolucao\n    INTO\n        v_id_livro,\n        v_data_prevista\n    FROM emprestimo\n    WHERE id_emprestimo = NEW.id_emprestimo;\n\n    UPDATE livro\n    SET quantidade_disponivel =\n        LEAST(\n            quantidade_total,\n            quantidade_disponivel + 1\n        )\n    WHERE id_livro = v_id_livro;\n\n    UPDATE emprestimo\n    SET status = 'Devolvido'\n    WHERE id_emprestimo = NEW.id_emprestimo;\n\n    SET v_dias_atraso =\n        GREATEST(\n            DATEDIFF(\n                NEW.data_devolucao,\n                v_data_prevista\n            ),\n            0\n        );\n\n    IF v_dias_atraso > 0 THEN\n        INSERT INTO multa (\n            id_emprestimo,\n            valor_total,\n            valor_pago,\n            data_geracao,\n            status\n        ) VALUES (\n            NEW.id_emprestimo,\n            v_dias_atraso * v_valor_diario,\n            0.00,\n            NEW.data_devolucao,\n            'Pendente'\n        );\n    END IF;\nEND$$\n\n-- Valida pagamentos parciais ou integrais\nCREATE TRIGGER trg_validar_pagamento_multa\nBEFORE INSERT ON pagamento_multa\nFOR EACH ROW\nBEGIN\n    DECLARE v_multa_existe INT DEFAULT 0;\n    DECLARE v_valor_total DECIMAL(10,2) DEFAULT 0.00;\n    DECLARE v_valor_pago_atual DECIMAL(10,2) DEFAULT 0.00;\n    DECLARE v_status VARCHAR(30);\n\n    SELECT\n        COUNT(*),\n        COALESCE(MAX(valor_total), 0.00),\n        COALESCE(MAX(valor_pago), 0.00),\n        MAX(status)\n    INTO\n        v_multa_existe,\n        v_valor_total,\n        v_valor_pago_atual,\n        v_status\n    FROM multa\n    WHERE id_multa = NEW.id_multa;\n\n    IF v_multa_existe = 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Multa não encontrada.';\n    END IF;\n\n    IF NEW.valor_pago <= 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'O valor do pagamento deve ser positivo.';\n    END IF;\n\n    IF v_status IN ('Paga', 'Cancelada') THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Esta multa não pode receber novos pagamentos.';\n    END IF;\n\n    IF NEW.valor_pago >\n       (v_valor_total - v_valor_pago_atual) THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'O pagamento ultrapassa o saldo da multa.';\n    END IF;\nEND$$\n\n-- Atualiza o valor pago e o status da multa\nCREATE TRIGGER trg_atualizar_multa_pagamento\nAFTER INSERT ON pagamento_multa\nFOR EACH ROW\nBEGIN\n    DECLARE v_valor_total DECIMAL(10,2);\n    DECLARE v_valor_pago_atual DECIMAL(10,2);\n    DECLARE v_novo_valor_pago DECIMAL(10,2);\n\n    SELECT\n        valor_total,\n        valor_pago\n    INTO\n        v_valor_total,\n        v_valor_pago_atual\n    FROM multa\n    WHERE id_multa = NEW.id_multa;\n\n    SET v_novo_valor_pago =\n        v_valor_pago_atual + NEW.valor_pago;\n\n    UPDATE multa\n    SET\n        valor_pago = v_novo_valor_pago,\n        status = CASE\n            WHEN v_novo_valor_pago >= valor_total\n                THEN 'Paga'\n            ELSE 'Parcialmente paga'\n        END\n    WHERE id_multa = NEW.id_multa;\nEND$$\n\nDELIMITER ;\n\n-- ============================================================\n-- 3. DADOS DE EXEMPLO\n-- ============================================================\n\nINSERT INTO editora (\n    nome,\n    cidade,\n    telefone\n) VALUES\n(\n    'Editora Acadêmica',\n    'São Paulo',\n    '(11) 3333-1111'\n),\n(\n    'Editora Saber',\n    'Rio de Janeiro',\n    '(21) 3333-2222'\n);\n\nINSERT INTO categoria (\n    nome,\n    descricao\n) VALUES\n(\n    'Tecnologia',\n    'Livros de computação e tecnologia'\n),\n(\n    'Literatura',\n    'Obras literárias nacionais e estrangeiras'\n),\n(\n    'Ciências',\n    'Livros das áreas de ciências exatas e naturais'\n);\n\nINSERT INTO autor (\n    nome,\n    nacionalidade,\n    data_nascimento\n) VALUES\n(\n    'Marcos Silva',\n    'Brasileira',\n    '1975-03-12'\n),\n(\n    'Luciana Costa',\n    'Brasileira',\n    '1980-07-21'\n),\n(\n    'Machado de Assis',\n    'Brasileira',\n    '1839-06-21'\n),\n(\n    'Paulo Almeida',\n    'Brasileira',\n    '1968-11-10'\n);\n\nINSERT INTO livro (\n    titulo,\n    isbn,\n    ano_publicacao,\n    quantidade_total,\n    quantidade_disponivel,\n    id_editora,\n    id_categoria\n) VALUES\n(\n    'Banco de Dados Modernos',\n    '978-65-00001-01-1',\n    2024,\n    3,\n    3,\n    1,\n    1\n),\n(\n    'Dom Casmurro',\n    '978-65-00001-02-8',\n    2020,\n    2,\n    2,\n    2,\n    2\n),\n(\n    'Fundamentos de Física',\n    '978-65-00001-03-5',\n    2023,\n    1,\n    1,\n    1,\n    3\n);\n\nINSERT INTO livro_autor (\n    id_livro,\n    id_autor\n) VALUES\n(1, 1),\n(1, 2),\n(2, 3),\n(3, 4);\n\nINSERT INTO usuario (\n    nome,\n    email,\n    matricula,\n    tipo_usuario,\n    telefone,\n    situacao\n) VALUES\n(\n    'Ana Beatriz Souza',\n    'ana.souza@universidade.edu.br',\n    'ALU2026001',\n    'Aluno',\n    '(65) 99999-1111',\n    'Ativo'\n),\n(\n    'Carlos Henrique Lima',\n    'carlos.lima@universidade.edu.br',\n    'PROF2026001',\n    'Professor',\n    '(65) 99999-2222',\n    'Ativo'\n),\n(\n    'Mariana Alves',\n    'mariana.alves@universidade.edu.br',\n    'ALU2026002',\n    'Aluno',\n    '(65) 99999-3333',\n    'Ativo'\n);\n\n-- Empréstimo 1: devolvido dentro do prazo\nINSERT INTO emprestimo (\n    id_usuario,\n    id_livro,\n    data_emprestimo,\n    data_prevista_devolucao,\n    status\n) VALUES (\n    1,\n    1,\n    '2026-06-01',\n    '2026-06-08',\n    'Em andamento'\n);\n\n-- Empréstimo 2: devolvido com 5 dias de atraso\nINSERT INTO emprestimo (\n    id_usuario,\n    id_livro,\n    data_emprestimo,\n    data_prevista_devolucao,\n    status\n) VALUES (\n    2,\n    2,\n    '2026-06-01',\n    '2026-06-10',\n    'Em andamento'\n);\n\n-- Empréstimo 3: permanece em andamento\nINSERT INTO emprestimo (\n    id_usuario,\n    id_livro,\n    data_emprestimo,\n    data_prevista_devolucao,\n    status\n) VALUES (\n    3,\n    3,\n    '2026-06-20',\n    '2026-07-05',\n    'Em andamento'\n);\n\n-- Devolução sem atraso\nINSERT INTO devolucao (\n    id_emprestimo,\n    data_devolucao,\n    observacao\n) VALUES (\n    1,\n    '2026-06-07',\n    'Livro devolvido em bom estado.'\n);\n\n-- Devolução com 5 dias de atraso\nINSERT INTO devolucao (\n    id_emprestimo,\n    data_devolucao,\n    observacao\n) VALUES (\n    2,\n    '2026-06-15',\n    'Livro devolvido com atraso.'\n);\n\n-- Localiza a multa gerada automaticamente\nSELECT id_multa\nINTO @id_multa_atraso\nFROM multa\nWHERE id_emprestimo = 2;\n\n-- Pagamento em duas partes\nINSERT INTO pagamento_multa (\n    id_multa,\n    data_pagamento,\n    valor_pago,\n    forma_pagamento\n) VALUES\n(\n    @id_multa_atraso,\n    '2026-06-16',\n    4.00,\n    'PIX'\n),\n(\n    @id_multa_atraso,\n    '2026-06-18',\n    6.00,\n    'Cartão'\n);\n"

arquivo_sql = Path('/content/atividade2_biblioteca_universitaria.sql')
arquivo_sql.write_text(sql, encoding='utf-8')
print('Script salvo em:', arquivo_sql)
print('Tamanho:', arquivo_sql.stat().st_size, 'bytes')


Script salvo em: /content/atividade2_biblioteca_universitaria.sql
Tamanho: 17048 bytes


In [3]:
import subprocess
from pathlib import Path

arquivo_sql = Path('/content/atividade2_biblioteca_universitaria.sql')

with arquivo_sql.open('r', encoding='utf-8') as arquivo:
    processo = subprocess.run(
        ['mariadb', '-u', 'root', '--default-character-set=utf8mb4'],
        stdin=arquivo,
        text=True,
        capture_output=True
    )

print(processo.stdout)

if processo.returncode != 0:
    print('ERRO DO MYSQL/MARIADB:')
    print(processo.stderr)
    raise RuntimeError('A execução foi interrompida.')

print('Banco biblioteca_universitaria criado com sucesso.')



Banco biblioteca_universitaria criado com sucesso.


In [4]:
import subprocess

consultas = "USE biblioteca_universitaria;\n\nSELECT\n    'LIVROS, EDITORAS E CATEGORIAS' AS etapa;\n\nSELECT\n    l.id_livro,\n    l.titulo,\n    l.isbn,\n    l.ano_publicacao,\n    e.nome AS editora,\n    c.nome AS categoria,\n    l.quantidade_total,\n    l.quantidade_disponivel\nFROM livro AS l\nINNER JOIN editora AS e\n    ON e.id_editora = l.id_editora\nINNER JOIN categoria AS c\n    ON c.id_categoria = l.id_categoria\nORDER BY l.id_livro;\n\nSELECT\n    'AUTORES POR LIVRO' AS etapa;\n\nSELECT\n    l.titulo,\n    GROUP_CONCAT(\n        a.nome\n        ORDER BY a.nome\n        SEPARATOR ', '\n    ) AS autores\nFROM livro AS l\nINNER JOIN livro_autor AS la\n    ON la.id_livro = l.id_livro\nINNER JOIN autor AS a\n    ON a.id_autor = la.id_autor\nGROUP BY\n    l.id_livro,\n    l.titulo\nORDER BY l.titulo;\n\nSELECT\n    'EMPRÉSTIMOS E DEVOLUÇÕES' AS etapa;\n\nSELECT\n    em.id_emprestimo,\n    u.nome AS usuario,\n    u.tipo_usuario,\n    l.titulo,\n    em.data_emprestimo,\n    em.data_prevista_devolucao,\n    d.data_devolucao,\n    em.status\nFROM emprestimo AS em\nINNER JOIN usuario AS u\n    ON u.id_usuario = em.id_usuario\nINNER JOIN livro AS l\n    ON l.id_livro = em.id_livro\nLEFT JOIN devolucao AS d\n    ON d.id_emprestimo = em.id_emprestimo\nORDER BY em.id_emprestimo;\n\nSELECT\n    'MULTAS' AS etapa;\n\nSELECT\n    m.id_multa,\n    m.id_emprestimo,\n    m.valor_total,\n    m.valor_pago,\n    (m.valor_total - m.valor_pago) AS saldo,\n    m.data_geracao,\n    m.status\nFROM multa AS m\nORDER BY m.id_multa;\n\nSELECT\n    'PAGAMENTOS DA MULTA' AS etapa;\n\nSELECT\n    pm.id_pagamento,\n    pm.id_multa,\n    pm.data_pagamento,\n    pm.valor_pago,\n    pm.forma_pagamento\nFROM pagamento_multa AS pm\nORDER BY pm.id_pagamento;\n\nSELECT\n    'CONTAGEM FINAL' AS etapa;\n\nSELECT\n    'Editoras' AS tabela,\n    COUNT(*) AS total\nFROM editora\nUNION ALL\nSELECT 'Categorias', COUNT(*) FROM categoria\nUNION ALL\nSELECT 'Autores', COUNT(*) FROM autor\nUNION ALL\nSELECT 'Livros', COUNT(*) FROM livro\nUNION ALL\nSELECT 'Usuários', COUNT(*) FROM usuario\nUNION ALL\nSELECT 'Empréstimos', COUNT(*) FROM emprestimo\nUNION ALL\nSELECT 'Devoluções', COUNT(*) FROM devolucao\nUNION ALL\nSELECT 'Multas', COUNT(*) FROM multa\nUNION ALL\nSELECT 'Pagamentos', COUNT(*) FROM pagamento_multa;\n\nSELECT\n    'TRIGGERS CRIADAS' AS etapa;\n\nSHOW TRIGGERS\nFROM biblioteca_universitaria;\n"

resultado = subprocess.run(
    [
        'mariadb',
        '-u',
        'root',
        '--table',
        '--default-character-set=utf8mb4',
        '-e',
        consultas
    ],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print('ERRO DO MYSQL/MARIADB:')
    print(resultado.stderr)


+-------------------------------+
| etapa                         |
+-------------------------------+
| LIVROS, EDITORAS E CATEGORIAS |
+-------------------------------+
+----------+-------------------------+-------------------+----------------+--------------------+------------+------------------+-----------------------+
| id_livro | titulo                  | isbn              | ano_publicacao | editora            | categoria  | quantidade_total | quantidade_disponivel |
+----------+-------------------------+-------------------+----------------+--------------------+------------+------------------+-----------------------+
|        1 | Banco de Dados Modernos | 978-65-00001-01-1 |           2024 | Editora Acadêmica  | Tecnologia |                3 |                     3 |
|        2 | Dom Casmurro            | 978-65-00001-02-8 |           2020 | Editora Saber      | Literatura |                2 |                     2 |
|        3 | Fundamentos de Física   | 978-65-00001-03-5 |       

## Resultados esperados
- 2 editoras
- 3 categorias
- 4 autores
- 3 livros
- 3 usuários
- 3 empréstimos
- 2 devoluções
- 1 multa de R$ 10,00
- 2 pagamentos, quitando a multa
- 6 triggers criadas